# 09 Template แนวคิด KPI: Governance, Conflict และ Peace Analytics

Notebook นี้เป็น template สำหรับวางแผนการทำ analytics จากแหล่งข้อมูลของสถาบันพระปกเกล้า (KPI) ไม่ดาวน์โหลด ไม่ scrape และไม่สร้างข้อมูลตัวอย่าง

จึงไม่มีผลวิเคราะห์เชิงปริมาณ ไม่มี model และไม่เกี่ยวข้องกับข้อสรุป LABAI หลัก

## 00 แหล่งอ้างอิงและขอบเขต

แหล่งข้อมูลที่บันทึกไว้ในโครงการ:

- เว็บไซต์สถาบันพระปกเกล้า: https://www.kpi.ac.th/
- เอกสารสรุปโครงการ: `README.md`

ต้องตรวจ resource, เงื่อนไขการเข้าถึง, metadata และ schema ของไฟล์จริงก่อนเริ่ม analysis ทุกครั้ง


## 01 นำเข้าไลบรารี

ใช้ Path เพื่อตรวจเอกสารอ้างอิงในโครงการ และ pandas เพื่อแสดง checklist แบบตาราง ไม่มีการเชื่อมต่อเครือข่าย

In [1]:
from pathlib import Path

import pandas as pd

## 02 ค้นหา root ของโครงการ

ตรวจหา `README.md` และ `docs/` เพื่อให้ template เปิดได้จากตำแหน่ง notebook ที่ต่างกัน

In [2]:
def find_project_root():
    current_path = Path.cwd().resolve()
    candidate_paths = [current_path]
    candidate_paths.extend(current_path.parents)

    for candidate_path in candidate_paths:
        if (candidate_path / "README.md").exists() and (candidate_path / "docs").exists():
            return candidate_path

    raise FileNotFoundError("ไม่พบ root ของโครงการที่มี docs")


PROJECT_ROOT = find_project_root()


## 03 ตรวจเอกสารอ้างอิงในโครงการ

การตรวจนี้ยืนยันเฉพาะว่าเอกสารสรุปและ manifest มีอยู่ ไม่ได้ยืนยันว่ามีไฟล์ KPI พร้อมวิเคราะห์

In [3]:
reference_paths = [
    PROJECT_ROOT / "README.md",
]

kpi_reference_status_df = pd.DataFrame(
    {
        "เอกสาร": [str(path.relative_to(PROJECT_ROOT)) for path in reference_paths],
        "มีอยู่": [path.exists() for path in reference_paths],
    }
)

if not kpi_reference_status_df["มีอยู่"].all():
    raise FileNotFoundError("ไม่พบ README ที่ต้องใช้ใน template")

kpi_reference_status_df


,เอกสาร,มีอยู่
0,README.md,True
1,docs\github-upload-manifest.md,True


## 04 สิ่งที่ต้องตรวจเมื่อได้รับข้อมูลจริง

ตารางนี้ระบุประเภทข้อมูลหรือ column ที่ควรตรวจ ไม่ใช่รายชื่อ schema ที่ยืนยันแล้วของ KPI

In [4]:
kpi_schema_checklist_df = pd.DataFrame(
    {
        "สิ่งที่ต้องตรวจ": [
            "รหัสรายการและแหล่งที่มา",
            "วันหรือช่วงเวลา",
            "พื้นที่และรหัสพื้นที่",
            "ประเภทประเด็นหรือเหตุการณ์",
            "ข้อความบรรยายหรือรายละเอียด",
            "ตัวชี้วัด governance, conflict หรือ peace",
            "สถานะความครบถ้วนและเงื่อนไขการใช้ข้อมูล",
        ],
        "เหตุผล": [
            "ใช้ตรวจ duplicate และตรวจย้อนกลับไปยัง resource",
            "จำเป็นต่อการสร้าง timeline หรือการเปรียบเทียบตามเวลา",
            "จำเป็นต่อการเชื่อมแผนที่หรือเปรียบเทียบพื้นที่หลังยืนยันรหัส",
            "จำเป็นต่อการทำ descriptive count และการจัดกลุ่มเชิงเนื้อหา",
            "ใช้ได้กับ text analysis เฉพาะเมื่ออนุญาตและทำความสะอาดข้อความแล้ว",
            "ต้องยืนยันนิยาม หน่วย และวิธีคำนวณก่อนทำ scorecard หรือ model",
            "ป้องกันการใช้ข้อมูลเกินวัตถุประสงค์หรือเชื่อมข้อมูลโดยไม่มีสิทธิ์",
        ],
    }
)
kpi_schema_checklist_df

,สิ่งที่ต้องตรวจ,เหตุผล
0,รหัสรายการและแหล่งที่มา,ใช้ตรวจ duplicate และตรวจย้อนกลับไปยัง resource
1,วันหรือช่วงเวลา,จำเป็นต่อการสร้าง timeline หรือการเปรียบเทียบต...
2,พื้นที่และรหัสพื้นที่,จำเป็นต่อการเชื่อมแผนที่หรือเปรียบเทียบพื้นที่...
3,ประเภทประเด็นหรือเหตุการณ์,จำเป็นต่อการทำ descriptive count และการจัดกลุ่...
4,ข้อความบรรยายหรือรายละเอียด,ใช้ได้กับ text analysis เฉพาะเมื่ออนุญาตและทำค...
5,"ตัวชี้วัด governance, conflict หรือ peace",ต้องยืนยันนิยาม หน่วย และวิธีคำนวณก่อนทำ score...
6,สถานะความครบถ้วนและเงื่อนไขการใช้ข้อมูล,ป้องกันการใช้ข้อมูลเกินวัตถุประสงค์หรือเชื่อมข...


## 05 แผน analytics ที่เป็นไปได้หลังตรวจ schema

แต่ละแนวคิดเป็นเงื่อนไขในอนาคต ต้องเริ่มหลังมีไฟล์จริงและมี metadata ยืนยัน ไม่ใช่ผลที่ทำแล้วใน template นี้

In [5]:
kpi_analytics_plan_df = pd.DataFrame(
    {
        "ประเภท analytics": ["Descriptive", "Diagnostic", "Predictive", "Prescriptive"],
        "ตัวอย่างคำถาม": [
            "ข้อมูลบันทึกตามเวลา พื้นที่ หรือประเภทใดบ้าง",
            "ความแตกต่างที่เห็นอาจเกี่ยวข้องกับความครบถ้วนของการบันทึกหรือไม่",
            "มี label และข้อมูลตามเวลาที่เพียงพอสำหรับ baseline หรือไม่",
            "ควรเก็บ metadata หรือ variables ใดเพิ่มก่อนใช้ผลเพื่อการตัดสินใจ",
        ],
        "เงื่อนไขก่อนทำ": [
            "ยืนยันวัน พื้นที่ ประเภท และการนับหน่วยรายการ",
            "ตรวจ missing, duplicate, นิยามตัวชี้วัด และความสอดคล้องพื้นที่",
            "มี target ที่ยืนยันได้, sample เพียงพอ และไม่มี leakage",
            "แยกผลที่สังเกตได้ออกจากผล model และยืนยันบริบทกับผู้ดูแลข้อมูล",
        ],
    }
)
kpi_analytics_plan_df

,ประเภท analytics,ตัวอย่างคำถาม,เงื่อนไขก่อนทำ
0,Descriptive,ข้อมูลบันทึกตามเวลา พื้นที่ หรือประเภทใดบ้าง,ยืนยันวัน พื้นที่ ประเภท และการนับหน่วยรายการ
1,Diagnostic,ความแตกต่างที่เห็นอาจเกี่ยวข้องกับความครบถ้วนข...,"ตรวจ missing, duplicate, นิยามตัวชี้วัด และควา..."
2,Predictive,มี label และข้อมูลตามเวลาที่เพียงพอสำหรับ base...,"มี target ที่ยืนยันได้, sample เพียงพอ และไม่ม..."
3,Prescriptive,ควรเก็บ metadata หรือ variables ใดเพิ่มก่อนใช้...,แยกผลที่สังเกตได้ออกจากผล model และยืนยันบริบท...


## 06 ข้อจำกัดของ template

- ไม่มี raw file KPI ใน `data/raw/` ในรอบนี้
- ไม่ตรวจ schema, row count หรือ column name ของ KPI เพราะยังไม่มีไฟล์จริง
- ไม่ scrape หน้าเว็บและไม่ใช้ข้อมูลสมมติ
- ไม่ควรตีความตาราง planning เป็นผล governance, conflict หรือ peace ของพื้นที่ใด

## 07 สถานะการรัน template

cell นี้บันทึกว่าสามารถรัน template ได้โดยไม่ต้องใช้ network หรือ raw data และผลลัพธ์เป็นเพียง checklist สำหรับขั้นตอนถัดไป

In [6]:
kpi_template_status_df = pd.DataFrame(
    {
        "รายการ": ["การเข้าถึง raw data", "การสร้างข้อมูล", "การสร้าง model", "ผลลัพธ์ของ notebook"],
        "สถานะ": [
            "ไม่ดำเนินการใน template",
            "ไม่สร้างข้อมูลสมมติ",
            "ไม่สร้าง model",
            "checklist และแผนตรวจ schema ก่อนเริ่มงานจริง",
        ],
    }
)
kpi_template_status_df

,รายการ,สถานะ
0,การเข้าถึง raw data,ไม่ดำเนินการใน template
1,การสร้างข้อมูล,ไม่สร้างข้อมูลสมมติ
2,การสร้าง model,ไม่สร้าง model
3,ผลลัพธ์ของ notebook,checklist และแผนตรวจ schema ก่อนเริ่มงานจริง


## 08 แหล่งข้อมูล

- สถาบันพระปกเกล้า: https://www.kpi.ac.th/
- README ของโครงการ: `README.md`
